# Batch GLI: interactive folder → aggregate CSV

1. **Pick a root folder** (browse or type path) — all `.xlsx` / `.xls` files are found (optionally in subfolders).
2. Compute **GLI** predicted, **LLN**, ULN, z-score, % predicted per parameter.
3. **Classify** each row: Normal / Obstruction / Restriction / Mixed (FEV₁/FVC + TLC vs LLN).
4. Save two CSVs: **full** (all original columns + GLI) and **summary** (test date, age, key measures, LLN, classification only).

Run the **setup** cell, then the **folder picker** cell. Optional summary cells below.

In [1]:
from datetime import datetime
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Project root (parent of notebooks/)
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "backend").exists():
    PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.app.services.batch_service import (
    build_compact_summary,
    discover_excel_files,
    process_excel_paths,
    read_pft_excel,
    save_batch_csv_outputs,
)
from backend.app.services.fields import suggest_mapping
from backend.app.services.gli_service import gli_service

# Warm-load GLI spline tables (requires data/reference/*.xlsx)
_ = gli_service.tlc_splines
_ = gli_service.spiro_splines

MODULES = ["spirometry", "lung_volumes"]
AUTO_MAPPING_PER_FILE = True
DEFAULT_OUTPUT_DIR = PROJECT_ROOT / "data/output"

# Edit these if you prefer typing a path instead of the folder picker
BATCH_ROOT = str(PROJECT_ROOT / "data/samples")
BATCH_RECURSIVE = True
BATCH_CSV_NAME = "aggregated_PFT_GLI.csv"

combined_df: pd.DataFrame | None = None

print("GLI reference tables loaded.")
print("Reference data:", PROJECT_ROOT / "data/reference")

GLI reference tables loaded.
Reference data: /Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/data/reference


## Select folder → run → CSV

**Run the cell below.** On macOS, a **Finder** folder picker opens (no Jupyter widgets).

- Cancel the picker → uses `BATCH_ROOT` from the setup cell.
- Or edit `BATCH_ROOT` there and run `run_gli_batch()` in a new cell.

In [2]:
def pick_folder(initial: str | None = None) -> str | None:
    """Folder dialog without ipywidgets. macOS: Finder picker; else tkinter or typed path."""
    import subprocess
    import sys

    start = initial or BATCH_ROOT or str(PROJECT_ROOT)

    if sys.platform == "darwin":
        script = (
            'POSIX path of (choose folder with prompt '
            '"Select root folder containing PFT Excel files")'
        )
        proc = subprocess.run(
            ["osascript", "-e", script],
            capture_output=True,
            text=True,
        )
        if proc.returncode == 0 and proc.stdout.strip():
            return proc.stdout.strip()
        if proc.returncode != 0:
            return None  # user cancelled Finder dialog

    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        chosen = filedialog.askdirectory(
            initialdir=start,
            title="Select root folder containing PFT Excel files",
        )
        root.destroy()
        if chosen:
            return chosen
    except Exception:
        pass

    print(f"Type folder path and press Enter (default: {start})")
    typed = input("Folder: ").strip()
    return typed or start


def run_gli_batch(
    root: str | Path | None = None,
    *,
    recursive: bool | None = None,
    csv_name: str | None = None,
) -> pd.DataFrame | None:
    """Find all Excel under root, compute GLI, write aggregated CSV."""
    global combined_df, BATCH_ROOT, BATCH_RECURSIVE, BATCH_CSV_NAME

    root_path = Path((root or BATCH_ROOT).strip()).expanduser()
    do_recursive = BATCH_RECURSIVE if recursive is None else recursive
    out_name = csv_name or BATCH_CSV_NAME

    BATCH_ROOT = str(root_path)
    BATCH_RECURSIVE = do_recursive
    BATCH_CSV_NAME = out_name

    if not root_path.is_dir():
        print(f"ERROR: Folder not found:\n  {root_path}")
        return None

    files = discover_excel_files(root_path, recursive=do_recursive)
    if not files:
        print(f"No Excel files under:\n  {root_path}")
        return None

    print(f"Root: {root_path}")
    print(f"Found {len(files)} file(s):")
    for f in files:
        print(f"  • {f.relative_to(root_path)}")

    print("\nRunning GLI (this may take a minute)…")
    combined_df, row_results, skipped = process_excel_paths(
        files,
        mapping=None,
        modules=MODULES,
        auto_mapping_per_file=AUTO_MAPPING_PER_FILE,
    )

    out_path = DEFAULT_OUTPUT_DIR / out_name.strip()
    full_path, summary_path = save_batch_csv_outputs(combined_df, out_path)
    compact_df = build_compact_summary(combined_df)

    print("\n✓ Done")
    print(f"  Rows: {len(combined_df)}  |  Skipped/incomplete: {skipped}")
    if "GLI_PFT_Pattern" in combined_df.columns:
        print("\n  Pattern counts:")
        print(combined_df["GLI_PFT_Pattern"].value_counts().to_string())
    print(f"\n  Full CSV ({len(combined_df.columns)} columns):\n  {full_path}")
    print(f"  Summary CSV ({len(compact_df.columns)} columns):\n  {summary_path}")
    display(compact_df.head(5))
    return combined_df


# Run this cell → folder picker opens, then batch runs
print("Opening folder picker… (Cancel uses BATCH_ROOT from the setup cell)")
_selected = pick_folder()
if _selected:
    print(f"Selected: {_selected}\n")
    run_gli_batch(root=_selected)
else:
    print(f"No folder picked — using BATCH_ROOT:\n  {BATCH_ROOT}\n")
    run_gli_batch()

Opening folder picker… (Cancel uses BATCH_ROOT from the setup cell)
Selected: /Users/alirezakeshavarzian/ThesisProject/Oscillometry-GUI/Dataset/PFT/

Root: /Users/alirezakeshavarzian/ThesisProject/Oscillometry-GUI/Dataset/PFT
Found 291 file(s):
  • BMT/Archive/BMT PFT 20200720 to 20200828.xlsx
  • BMT/Archive/BMT PFT 20200831 to 20201002.xlsx
  • BMT/Archive/BMT PFT 20201005 to 20201030.xlsx
  • BMT/Archive/BMT PFT 20201102 to 20201106.xlsx
  • BMT/Archive/BMT PFT 20201109 to 20201113.xlsx
  • BMT/Archive/BMT PFT 20201116 to 20201120.xlsx
  • BMT/Archive/BMT PFT 20201123 to 20201127.xlsx
  • BMT/Archive/BMT PFT 20201130 to 20201204.xlsx
  • BMT/Archive/BMT PFT 20201207 to 20201211.xlsx
  • BMT/Archive/BMT PFT 20201214 to 20201218.xlsx
  • BMT/Archive/BMT PFT 20201221 to 20201225.xlsx
  • BMT/Archive/BMT PFT 20210104 to 20210108.xlsx
  • BMT/Archive/BMT PFT 20210111 to 20210115.xlsx
  • BMT/Archive/BMT PFT 20210118 to 20210122.xlsx
  • BMT/Archive/BMT PFT 20210125 to 20210129.xlsx
  • B

/Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/backend/app/services/batch_service.py:534: UserWarning: Discarding nonzero nanoseconds in conversion.
  return pd.to_datetime(val).to_pydatetime()
/Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/backend/app/services/batch_service.py:534: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(val).to_pydatetime()
/Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/backend/app/services/batch_service.py:534: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(val).to_pydatetime()
/Users/alirezakeshavarzian/ThesisProject/LLN-GLI-Pulmonary/backend/app/services/batch_service.py:534: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the defau

ImportError: `Import xlrd` failed. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install the xlrd package.

## Preview mapping (first Excel file in folder)

Run after setting the folder above, or set `preview_root` manually.

In [ ]:
preview_root = Path(BATCH_ROOT).expanduser()
preview_files = (
    discover_excel_files(preview_root, recursive=BATCH_RECURSIVE)
    if preview_root.is_dir()
    else []
)

if preview_files:
    preview_path = preview_files[0]
    preview_df = read_pft_excel(preview_path)
    suggested = suggest_mapping([str(c) for c in preview_df.columns])
    print(f"First file: {preview_path.name} ({len(preview_df.columns)} columns)")
    print("\nSuggested mapping:")
    for key, col in suggested.model_dump().items():
        if col:
            print(f"  {key:12} → {col}")
else:
    print("Set a valid root folder in the interactive cell above.")

## Full results (after interactive run)

Run the interactive cell first. `combined_df` holds all rows.

In [ ]:
if combined_df is None or combined_df.empty:
    print("Run the folder picker cell above first.")
else:
    print(f"Rows: {len(combined_df)}  |  Columns: {len(combined_df.columns)}")
    display(combined_df.head(10))

## Summary: PFT pattern classification

In [ ]:
if combined_df is None or combined_df.empty:
    print("Run the interactive batch first.")
elif "GLI_PFT_Pattern" in combined_df.columns:
    display(
        combined_df["GLI_PFT_Pattern"]
        .value_counts(dropna=False)
        .rename("count")
        .to_frame()
    )

    status_cols = [c for c in combined_df.columns if c.endswith("_Status")]
    below_counts = {}
    for col in status_cols:
        param = col.replace("GLI_", "").replace("_Status", "")
        below_counts[param] = (combined_df[col] == "below_lln").sum()
    if below_counts:
        print("\nRows below LLN by parameter:")
        display(pd.Series(below_counts, name="below_lln_count").sort_values(ascending=False))

## Summary preview (same columns as `*_summary.csv`)

In [ ]:
if combined_df is None:
    print("No data yet — run the batch cell first.")
else:
    display(build_compact_summary(combined_df).head(20))

## Re-save CSV (optional)

Change filename if needed; uses the same `combined_df` from the interactive run.

In [ ]:
if combined_df is None or combined_df.empty:
    print("Nothing to save — run the batch cell first.")
else:
    full_path, summary_path = save_batch_csv_outputs(
        combined_df, DEFAULT_OUTPUT_DIR / BATCH_CSV_NAME.strip()
    )
    print(f"Full:    {full_path}")
    print(f"Summary: {summary_path}")